# Сборка датасета из бакета

Гоняет стадию `build` прямо в облаке: читает сырьё из `raw/`, приводит кадры к
единому формату и складывает новый датасет в `curated/`.

Обработка идёт здесь, а не на ноутбуке, по простой причине: сырья 9 ГБ, и
внутри облака трафик до бакета не тарифицируется, а наружу — платный.

## Что гарантировано

`raw/` и `manifest/` открываются **только на чтение**. Хранилище выдаётся коду с
единственным разрешённым префиксом записи, и попытка выйти за него поднимает
исключение. Старые датасеты потрогать невозможно даже по ошибке.

Новый датасет называется по хешу рецепта, так что разные рецепты дают разные
каталоги и не затирают друг друга.

## 1. Подключение кода

Ядру нужен пакет `data` из репозитория. Способы, по убыванию удобства:

1. **клон репозитория** в проект DataSphere — тогда код обновляется `git pull`;
2. **загрузка каталога** `data/` через файловый браузер DataSphere;
3. `sys.path` вручную, если положили куда-то ещё.

Ячейка ищет пакет в очевидных местах и говорит, чего не хватает.

In [ ]:
import os, sys
from pathlib import Path

CANDIDATES = [
    Path.cwd(), Path.cwd().parent,
    Path("/home/jupyter/OcclusionNet"),
    Path("/home/jupyter/work/OcclusionNet"),
    Path("/home/jupyter/work/resources/OcclusionNet"),
]

root = next((p for p in CANDIDATES if (p / "data" / "build.py").exists()), None)
if root is None:
    raise RuntimeError(
        "не нашёл пакет data. Положите репозиторий рядом с блокнотом:\n"
        "  git clone <repo> /home/jupyter/OcclusionNet\n"
        "или загрузите каталог data/ (128 КБ) через файловый браузер.\n"
        "Искал в:\n  " + "\n  ".join(str(p) for p in CANDIDATES))

sys.path.insert(0, str(root))
from data.build import run
from data import s3io

print("код взят из", root)

## 2. Секреты и бакет

Те же переменные, что в остальных блокнотах: на удалённом ядре они заводятся
секретами проекта, после чего ядро надо перезапустить.

In [ ]:
BUCKET = os.environ.get("DATASETS_BUCKET", "occlusionnet-clearml-b052c3-datasets")

missing = [k for k in ("S3_KEY", "S3_SECRET") if not os.environ.get(k)]
if missing:
    raise RuntimeError(f"нет ключей {missing}: секреты проекта DataSphere "
                       "и перезапуск ядра")

store = s3io.S3Store(s3io.client(), BUCKET)     # без write_prefix: только чтение
total = sum(o["Size"] for o in store.list("raw/"))
print(f"бакет {BUCKET}")
print(f"сырья в raw/: {total/1024**3:.2f} ГБ")
for o in sorted(store.list("manifest/"), key=lambda x: x["Key"]):
    print(f"  {o['Key']:<44} {o['Size']/1024:8.0f} КБ")

## 3. Сухой прогон

Считает план и калибровку, ничего не пишет. Смотреть надо на три вещи:

* состав выборок — не пуст ли `val`;
* сиквенсов в нескольких выборках должно быть ноль;
* подобранное размытие на источник.

In [ ]:
RECIPE = root / "data" / "recipe.example.yaml"

summary = run(recipe_path=RECIPE, bucket=BUCKET, dry_run=True)
summary

## 4. Сборка

Читает кадры из архивов range-запросами, приводит к 512x512, пишет шарды и
выгружает их в `curated/`.

По времени: около семи тысяч кадров, на каждый декодирование, кроп, ресайз и
кодирование. На четырёх ядрах это порядка десяти минут. Чтение сырья идёт
внутри облака и трафиком не тарифицируется.

In [ ]:
result = run(recipe_path=RECIPE, bucket=BUCKET)
result

## 5. Что получилось в бакете

In [ ]:
import io, pandas as pd, pyarrow.parquet as pq

prefix = result["write_prefix"]
objs = sorted(store.list(prefix), key=lambda o: o["Key"])
print(f"{prefix}: объектов {len(objs)}, "
      f"{sum(o['Size'] for o in objs)/1024**3:.2f} ГБ")
for o in objs[:20]:
    print(f"  {o['Key'][len(prefix):]:<44} {o['Size']/1024**2:8.1f} МБ")

cur = pq.read_table(io.BytesIO(store.read(prefix + "manifest.parquet"))).to_pandas()
cur["метка"] = cur.labels.apply(lambda l: "+".join(sorted(l)) or "clean")
display(cur.groupby(["split", "метка"]).size().rename("кадров").to_frame())

print("\nsanity: сырьё не изменилось")
print(f"  объектов в raw/: {len(store.list('raw/'))}, "
      f"{sum(o['Size'] for o in store.list('raw/'))/1024**3:.2f} ГБ")

## 6. Пара кадров из собранного датасета

Проверка глазами: то ли лежит в шардах, что мы думаем.

In [ ]:
import json, tarfile
import matplotlib.pyplot as plt
from PIL import Image

shard = next(o["Key"] for o in objs if o["Key"].endswith(".tar"))
blob = store.read(shard)
with tarfile.open(fileobj=io.BytesIO(blob)) as t:
    names = [n for n in t.getnames() if n.endswith(".jpg")][:4]
    fig, axes = plt.subplots(1, len(names), figsize=(3.4 * len(names), 3.8))
    axes = [axes] if len(names) == 1 else list(axes)
    for ax, n in zip(axes, names):
        img = Image.open(io.BytesIO(t.extractfile(n).read()))
        meta = json.loads(t.extractfile(n[:-4] + ".json").read())
        ax.imshow(img); ax.set_axis_off()
        ax.set_title(f"{meta['source']}\n{'+'.join(meta['labels']) or 'clean'}\n"
                     f"{img.size[0]}x{img.size[1]} · sigma {meta['sigma']}",
                     fontsize=8, color="#898781")
fig.suptitle(shard.split("/")[-1], fontsize=11, fontweight="bold", y=1.03)
fig.tight_layout(); plt.show()

## 7. Версия в ClearML

Регистрируем ссылками, а не копией: шарды уже лежат в бакете, дублировать их
внутрь хранилища ClearML незачем.

In [ ]:
import subprocess

LIBS = os.path.expanduser("~/dslibs")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "--target", LIBS, "clearml"])
if LIBS not in sys.path:
    sys.path.insert(0, LIBS)

tpl = """sdk {
    aws {
        s3 {
            region: "ru-central1"
            credentials: [
                { host: "storage.yandexcloud.net:443"
                  key: "@KEY@"
                  secret: "@SECRET@"
                  secure: true }
            ]
        }
    }
}
"""
conf = os.path.expanduser("~/clearml.conf")
with open(conf, "w") as fh:
    fh.write(tpl.replace("@KEY@", os.environ["S3_KEY"])
                .replace("@SECRET@", os.environ["S3_SECRET"]))
os.chmod(conf, 0o600)

from clearml import Dataset, Task

os.environ.setdefault("CLEARML_WEB_HOST",   "https://app.occlusionnet.duckdns.org")
os.environ.setdefault("CLEARML_API_HOST",   "https://api.occlusionnet.duckdns.org")
os.environ.setdefault("CLEARML_FILES_HOST", "https://files.occlusionnet.duckdns.org")

BASE = f"s3://storage.yandexcloud.net:443/{BUCKET}"

ds = Dataset.create(dataset_project="OcclusionNet",
                    dataset_name="occlusionnet",
                    dataset_version=result["version"].split("-")[-1],
                    output_uri=BASE)
ds.add_external_files(source_url=f"{BASE}/{prefix}", wildcard="*.tar", recursive=True)
ds.add_external_files(source_url=f"{BASE}/{prefix}", wildcard="manifest.parquet")
ds.upload()
ds.finalize()
print("датасет зарегистрирован:", ds.id)